In [15]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

CAP = 24
YE = pd.Timestamp("2025-12-31")
FY_START = pd.Timestamp("2025-01-01")
DATES = ["referral_date", "intake_date", "admission_date", "discharge_date"]

df = pd.read_csv("harbourview_residents.csv")
for c in DATES:
    df[c] = pd.to_datetime(df[c])
AS_OF = df[DATES].max().max()  # 2026-01-15

df["outcome"] = df["current_status"].map(
    lambda s: "ongoing" if s.startswith("phase")
    else "completed" if s in ("alumni", "aftercare")
    else "early_discharge" if s == "discharged"
    else "not_admitted")
bad_discharge = (df.outcome == "early_discharge") & df.discharge_date.isna()

# Funnel and Time in stage

In [16]:
f25 = df[df.referral_date.dt.year == 2025].copy()
f25["reached_intake"] = (f25.intake_date <= YE) | (f25.admission_date <= YE)
f25["admitted"] = f25.admission_date <= YE

n_ref = len(f25)
n_int = int(f25.reached_intake.sum())
n_adm = int(f25.admitted.sum())
in_progress = int(((f25.referral_date >= "2025-12-01") & ~f25.reached_intake).sum())

In [10]:
df["reached_intake"] = df.intake_date.notna() | df.admission_date.notna()
f = df.groupby("fiscal_year").agg(referred=("resident_id","size"),
                         intake=("reached_intake","sum"),
                         admitted=("admission_date","count"))
f["ref→intake %"] = (f.intake/f.referred*100).round(0)
f["intake→adm %"] = (f.admitted/f.intake*100).round(0)

/tmp/ipykernel_8091/4245763583.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["reached_intake"] = df.intake_date.notna() | df.admission_date.notna()


In [17]:
stages = ["Referred", "Reached intake", "Admitted"]
vals = [n_ref, n_int, n_adm]
labels = [f"{n_ref}", f"{n_int} ({n_int/n_ref:.0%} of referred)", f"{n_adm} ({n_adm/n_int:.0%} of intake)"]
fig = go.Figure(go.Bar(x=stages, y=vals, text=labels, textposition="outside"))
fig.update_layout(title=f"FY2025 funnel (n={n_ref} referrals, as of 2025-12-31)",
                  yaxis_title="Residents", plot_bgcolor="white")
fig.show()
print(f"In progress at year end (Dec referrals with no intake yet): {in_progress} - not counted as drop-offs")

In progress at year end (Dec referrals with no intake yet): 5 - not counted as drop-offs


In [18]:
t = f25.copy()
t["Referral → intake"] = (t.intake_date.where(t.intake_date <= YE) - t.referral_date).dt.days
t["Intake → admission"] = (t.admission_date.where(t.admission_date <= YE) - t.intake_date).dt.days
long = t.melt(value_vars=["Referral → intake", "Intake → admission"],
              var_name="step", value_name="days").dropna()
long = long[long.days >= 0]  # drop data-entry errors

stats = long.groupby("step", sort=False).days.agg(
    n="count", median="median", p90=lambda s: s.quantile(0.9)).round(1)

fig = px.box(long, x="step", y="days", points="outliers",
             title="FY2025 days between stages (line = median, ◆ = P90)")
fig.add_scatter(x=stats.index, y=stats.p90, mode="markers+text", name="P90",
                marker=dict(symbol="diamond", size=11, color="red"),
                text=[f"P90 {v:.0f}d" for v in stats.p90], textposition="middle right")
fig.update_layout(plot_bgcolor="white")
fig.show()

# Occupancy and length of stay

In [19]:
adm = df[df.admission_date.notna() & ~bad_discharge]
end = adm.discharge_date.fillna(AS_OF + pd.Timedelta(days=1))  # still in house -> open stay
days = pd.date_range(FY_START, YE)
census = pd.Series([((adm.admission_date <= d) & (end > d)).sum() for d in days], index=days)
monthly = census.resample("MS").mean()
occ = (monthly / CAP * 100).round(1)

fig = go.Figure(go.Bar(x=occ.index.strftime("%b"), y=occ.values,
                       text=[f"{v:.0f}%" for v in occ.values], textposition="outside"))
fig.add_hline(y=100, line_dash="dot", annotation_text=f"Capacity ({CAP} beds)")
fig.update_layout(title="FY2025 monthly occupancy (average daily census ÷ 24 beds)",
                  yaxis_title="Occupancy %", plot_bgcolor="white")
fig.show()
print(f"FY2025 average occupancy: {census.mean()/CAP:.0%}")

FY2025 average occupancy: 89%


In [20]:
a = df[df.admission_date.notna() & ~bad_discharge & (df.admission_date <= YE)].copy()
ended_2025 = a.discharge_date.between(FY_START, YE)
still_in = a.discharge_date.isna() | (a.discharge_date > YE)

a["group"] = None
a.loc[ended_2025 & (a.outcome == "completed"), "group"] = "Completed"
a.loc[ended_2025 & (a.outcome != "completed"), "group"] = "Discharged early"
a.loc[still_in, "group"] = "Ongoing at year end (so far)"
a = a[a.group.notna()]
a["los_days"] = (a.discharge_date.where(~still_in, YE) - a.admission_date).dt.days

fig = px.box(a, x="group", y="los_days", points="all",
             title="FY2025 length of stay (days)",
             category_orders={"group": ["Completed", "Discharged early", "Ongoing at year end (so far)"]})
fig.update_layout(plot_bgcolor="white", yaxis_title="Days")
fig.show()
a.groupby("group").los_days.agg(n="count", median="median", p90=lambda s: s.quantile(0.9)).round(0)

,n,median,p90
group,,,
Completed,41,130.0,213.0
Discharged early,39,48.0,100.0
Ongoing at year end (so far),18,69.0,141.0


# Honest funder outcomes

In [21]:
fin = df[df.discharge_date.notna()]
rows, summary = [], []
for col, months in [("followup_6m", 6), ("followup_12m", 12)]:
    due = fin[fin.discharge_date + pd.DateOffset(months=months) <= AS_OF]
    status = due[col].fillna("missing")
    for s, n in status.value_counts().items():
        rows.append({"checkpoint": f"{months} months (n={len(due)})", "status": s, "residents": n})
    eng = (status == "engaged_in_recovery").sum()
    reached = status.isin(["engaged_in_recovery", "relapse_reported"]).sum()
    summary.append({"checkpoint": f"{months}m", "n_due": len(due),
                    "best case: engaged / reached": f"{eng}/{reached} = {eng/reached:.0%}" if reached else "n/a",
                    "worst case: engaged / all due": f"{eng}/{len(due)} = {eng/len(due):.0%}" if len(due) else "n/a"})

fig = px.bar(pd.DataFrame(rows), x="checkpoint", y="residents", color="status", text="residents",
             title="Follow-up outcomes (lost to follow-up shown separately)")
fig.update_layout(plot_bgcolor="white")
fig.show()